# Train a Text-to-Emoji Model



## Text Encoder

To produce the text embedding, we can use a pre-trained CLIP model.

In [1]:
import torch
import torch.nn as nn
from transformers import CLIPTextModel, CLIPTokenizer


class TextEncoder(nn.Module):
    def __init__(self, model_name: str, device: str):
        super().__init__()
        self.model_name = model_name
        self.model = CLIPTextModel.from_pretrained(model_name).to(device)
        self.tokenizer = CLIPTokenizer.from_pretrained(model_name)
        self.device = device
        # Get the text embedding dimension from the config
        self.text_embed_dim = self.model.config.hidden_size

    def forward(self, text: str) -> torch.Tensor:
        tokens = self.tokenizer(text, padding=True, truncation=True, return_tensors="pt").to(self.device)
        return self.model(**tokens).pooler_output

/home/ubuntu/.local/lib/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(
/home/ubuntu/.local/lib/python3.11/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/home/ubuntu/.local/lib/python3.11/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


## Training

Now that the denoising model can handle text embeddings as an additional input, we also need to modify the training step to pass in the text embeddings. The main logic remains the same.

### Classifier-Free Guidance in Training: Text Dropout

With 20% of chance, the text embedding will be set to an empty embedding. The actual logic of dropout happens inside the denoising model.

In [2]:
import itertools
import torch
import torch.nn as nn
from torch.nn import MSELoss
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from typing import Dict

from lib_4_1.diffusion import forward_diffusion
from lib_4_1.bookkeeping import Bookkeeping
from lib_4_1.config import TrainingConfig

def train(
    config: TrainingConfig,
    model: nn.Module,
    text_encoder: TextEncoder,
    train_dataloader: DataLoader,
    val_dataloader: DataLoader,
    noise_schedule: Dict,
    optimizer: torch.optim.Optimizer,
    steps: int=100,
    silent: bool=False,
    bookkeeping: Bookkeeping=None
) -> float:
  device = config.device
  num_denoising_steps = config.num_denoising_steps
  
  model.train()
  if not silent:
    print("Training on device:", device)
  max_train_steps = steps

  loss = None
  progress_bar = tqdm(itertools.cycle(train_dataloader), total=max_train_steps, disable=silent)
  step = 0
  criterion = MSELoss()
  for batch in progress_bar:
    x_0 = batch[0]  # x_0 is the clean image to teach the model to generate
    text = batch[1]["text"]  # text is the caption of the image
    assert len(text) == x_0.shape[0]
    # assert the type of text is a list of strings
    x_0 = x_0.float().to(device)  # x_0 is the clean data to teach the model to generate
    optimizer.zero_grad()

    # Implement classifier-free guidance training
    # Randomly drop out text conditioning with 10% probability
    # The dropout is applied to the batch as a whole.
    # Alternatively, we could apply it to each image in the batch.
    text_drop_prob = 0.2
    true_noise = common_noise = torch.randn(x_0.shape).to(device)
    t = torch.randint(0, num_denoising_steps, (x_0.shape[0],), device=device).long()
    x_t, _ = forward_diffusion(x_0, t, noise_schedule, noise=common_noise)

    with torch.no_grad():
        text_embeddings = text_encoder(text)

    # A dropout is applied to the ``text_embeddings`` input:
    #   This means `predicted_noise` will be computed with 20% probability of the text embeddings being dropped out.
    #   The model learns to predict the noise both with and without the text embeddings.
    predicted_noise = model(t=t, x=x_t, text_embeddings=text_embeddings, p_uncond=text_drop_prob)

    loss = criterion(predicted_noise, true_noise)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1)  # try commenting it out
    optimizer.step()

    step += 1

    if not silent:
      progress_bar.set_postfix({"loss": loss.cpu().item()})

    if bookkeeping:
      bookkeeping.run_callbacks(config=config, step=step, loss=loss, optimizer=optimizer, val_dataloader=val_dataloader)

    if step >= max_train_steps:
      break

  return loss

### Captioned Image Dataset and Text Encoder

Let's create the main components of the model. Its has under 15M parameters, rather small.

For text encoder, we are using CLIP.

A good dataset is important for the performance of the model. To learn more about how to create a captioned image dataset, please refer to [this tutorial](Image%20Captioning%20Lesson.ipynb).

In [3]:
from torch import optim
from lib_4_1.data import load_data
from lib_4_1.model import create_unet_model
from lib_4_1.diffusion import create_noise_schedule

config = TrainingConfig(dataset="valhalla/emoji-dataset", caption_column="text", batch_size=16, resolution=32)
text_encoder = TextEncoder("openai/clip-vit-large-patch14", "cuda:0")
text_encoder.eval()
train_ds, val_ds = load_data(config)
noise_schedule = create_noise_schedule(n_T=config.num_denoising_steps, device=config.device)
denoising_model = create_unet_model(config, config.device)
optimizer = optim.AdamW(denoising_model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)

model params: 14.68 M


### A Quick Data Check

A quick check that the inputs and targets of a training example look good.

In [4]:
for x in train_ds:
    print(x[0].shape)
    print(x[1])
    break

torch.Size([3, 32, 32])
{'label': 0, 'text': 'men wrestling'}


Similarly, check to see if the mini-batch look good.

In [5]:
train_dataloader = DataLoader(train_ds, batch_size=config.batch_size, shuffle=True)
val_dataloader = DataLoader(val_ds, batch_size=config.batch_size, shuffle=False)
for x in train_dataloader:
    print(x[0].shape)
    print(x[1]["text"])
    text_embeddings = text_encoder(x[1]["text"])
    print(text_embeddings.shape)
    break

torch.Size([16, 3, 32, 32])
['man getting haircut type 4', 'man', 'toilet', 'flag for hungary', 'blond man type 5', 'runner', 'ok hand sign', 'monorail', 'gear', 'hammer', 'handball', 'man walking type 6', 'alarm clock', 'tennis racquet and ball', 'nerd face', 'flag for canary islands']
torch.Size([16, 768])


### We Train

We train 20,000 steps. This can take 20~30 minutes on a A10 instance on Lambda Labs.

In [6]:
%%time

train(
    config=config,
    model=denoising_model,
    text_encoder=text_encoder,
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    noise_schedule=noise_schedule,
    optimizer=optimizer,
    steps=10000,
    silent=False
)

Training on device: cuda


  0%|          | 0/10000 [00:00<?, ?it/s]

CPU times: user 17min 21s, sys: 2.01 s, total: 17min 23s
Wall time: 17min 23s


tensor(0.0122, device='cuda:0', grad_fn=<MseLossBackward0>)

### Save the Model

The checkpoint will be useful in the next tutorial, where we see our model it in action.

In [7]:
# save the model
torch.save(denoising_model.state_dict(), "denoising_model_emoji.pth")


## Generate images with text conditioning

In the next tutorial, we will use the updated sampling code to generate images with text conditioning. By varying the `guidance_scale` parameter, we can see how the text conditioning affects the generated images.